# V4 SFT — Fresh Fine-Tune on Qwen3-8B

**Fresh fine-tune** (NOT continuation of v2/v3 LoRA) with V4 schema dataset (~604 training examples).

V4 schema changes:
- `intent` is now an object: `{"type": "...", "retrieval": "..."}`
- Entities have `description` instead of `attributes` + inline `facts`
- ALL facts in top-level `facts[]` with `entity_id` key
- `topic.label` always present
- New detailed system prompt with CRITICAL RULES
- EXISTING NODES: type capitalized, facts as plain strings

## 1. Load Base Model (Fresh — NO LoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

# Load base model — FRESH, no LoRA checkpoint
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-8B",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

free, total = torch.cuda.mem_get_info(0)
print(f"VRAM: {(total-free)/1e9:.1f} GB used / {total/1e9:.1f} GB total")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 2. Load V4 Dataset

In [ ]:
from datasets import load_dataset
from pathlib import Path
import json

TD = Path(r"D:\Development\acervo-graph-model\training_data\v4")

train_dataset = load_dataset(
    "json", data_files=str(TD / "s1_v4_full_training.jsonl"), split="train"
)
print(f"Training: {len(train_dataset)} examples")

val_dataset = load_dataset(
    "json", data_files=str(TD / "s1_v4_full_validation.jsonl"), split="train"
)
print(f"Validation: {len(val_dataset)} examples")

# V4 schema checks
sample = json.loads(train_dataset[0]["messages"][2]["content"])
assert "intent" in sample and isinstance(sample["intent"], dict), "intent must be an object"
assert "type" in sample["intent"], "intent must have 'type'"
assert "retrieval" in sample["intent"], "intent must have 'retrieval'"
assert "topic" in sample and "label" in sample["topic"], "topic must have label"
print(f"\nV4 schema check OK:")
print(f"  intent: {sample['intent']}")
print(f"  topic: {sample['topic']}")

# Check entities use description (not attributes)
for ex in train_dataset:
    out = json.loads(ex["messages"][2]["content"])
    for e in out.get("entities", []):
        assert "description" in e, f"Entity missing description: {e}"
        assert "attributes" not in e, f"Entity has old attributes field: {e}"
        assert "facts" not in e, f"Entity has old inline facts: {e}"
    for f in out.get("facts", []):
        assert "entity_id" in f, f"Fact missing entity_id: {f}"
        assert f["entity_id"], f"Fact has empty entity_id: {f}"
print(f"  All entities use 'description' (not attributes/facts)")
print(f"  All facts use 'entity_id' (not entity), none are null")

# System prompt check
sys_prompt = train_dataset[0]["messages"][0]["content"]
assert "CRITICAL RULES" in sys_prompt, "System prompt missing CRITICAL RULES!"
assert "entity_id" in sys_prompt, "System prompt missing entity_id reference!"
print(f"  System prompt has CRITICAL RULES section: OK")
print(f"  System prompt length: {len(sys_prompt)} chars")

## 3. Format for Training

In [ ]:
def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return {"text": text}

train_formatted = train_dataset.map(format_example)
val_formatted = val_dataset.map(format_example)

print(f"Formatted {len(train_formatted)} train, {len(val_formatted)} val")

# Check token lengths
inner_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
lengths = [len(inner_tokenizer.encode(ex["text"])) for ex in train_formatted]
print(f"Token lengths: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)/len(lengths):.0f}")
over_limit = sum(1 for l in lengths if l > MAX_SEQ_LENGTH)
if over_limit:
    print(f"WARNING: {over_limit} examples exceed {MAX_SEQ_LENGTH} tokens!")
else:
    print(f"All examples fit within {MAX_SEQ_LENGTH} tokens")

## 4. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    args=SFTConfig(
        output_dir="./outputs/s1_sft_v4",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=20,
        num_train_epochs=4,
        learning_rate=3e-5,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=40,
        save_strategy="steps",
        save_steps=40,
        save_total_limit=5,
        optim="adamw_8bit",
        seed=42,
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        dataset_num_proc=None,
        dataloader_num_workers=0,
        report_to="none",
    ),
)

estimated_steps = len(train_formatted) * 4 // (2 * 4)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Training {len(train_formatted)} examples x 4 epochs")
print(f"Estimated steps: {estimated_steps}")

In [ ]:
stats = trainer.train()
print(f"\nTraining complete.")
print(f"  Total steps: {stats.global_step}")
print(f"  Train loss:  {stats.training_loss:.4f}")

In [1]:
from unsloth import FastLanguageModel
import torch, json, sys
from pathlib import Path

MAX_SEQ_LENGTH = 2048

# Load from final_lora — fresh kernel, no leftover state
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./outputs/s1_sft_v4/final_lora",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(model)
inner_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
print("Model loaded on", next(model.parameters()).device)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0410 19:11:29.943000 5728 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.18: Fast Qwen3 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 5070 Ti. Num GPUs = 1. Max memory: 15.92 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 399/399 [00:03<00:00, 105.32it/s]


unsloth/qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.18 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Model loaded on cuda:0


## 5. Quick Tests (V4 Schema)

In [2]:
import json
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "01_dataset"))
from schema_v4 import S1_V4_SYSTEM_PROMPT, validate_s1_v4

FastLanguageModel.for_inference(model)
inner_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
print("Model ready for inference.\n")

SYS = S1_V4_SYSTEM_PROMPT

test_cases = [
    {"label": "overview intent",
     "check": lambda p: p["intent"]["type"] == "overview",
     "messages": [{"role": "system", "content": SYS},
      {"role": "user", "content": 'EXISTING NODES:\n[{"id": "beacon", "label": "Beacon", "type": "Project", "facts": ["React+TypeScript frontend"]}]\n\nTOPIC HINT: unresolved\nCURRENT TOPIC: null\n\nPREVIOUS ASSISTANT: null\nUSER: What is this project about?'}]},

    {"label": "chat intent (greeting)",
     "check": lambda p: p["intent"]["type"] == "chat" and len(p["entities"]) == 0,
     "messages": [{"role": "system", "content": SYS},
      {"role": "user", "content": 'EXISTING NODES:\n[]\n\nTOPIC HINT: same\nCURRENT TOPIC: null\n\nPREVIOUS ASSISTANT: null\nUSER: Hola! Cómo andás?'}]},

    {"label": "facts with entity_id (existing node)",
     "check": lambda p: len(p["facts"]) > 0 and all(f["entity_id"] for f in p["facts"]),
     "messages": [{"role": "system", "content": SYS},
      {"role": "user", "content": 'EXISTING NODES:\n[{"id": "fondo_emergencia", "label": "Fondo de Emergencia", "type": "Concept", "facts": ["Objetivo: gastos imprevistos"]}]\n\nTOPIC HINT: same\nCURRENT TOPIC: finanzas personales\n\nPREVIOUS ASSISTANT: Dale, anoto.\nUSER: El fondo de emergencia ya tiene 3.500.000 ARS acumulados.'}]},

    {"label": "new entity with description + facts",
     "check": lambda p: len(p["entities"]) > 0 and all("description" in e for e in p["entities"]) and all("attributes" not in e for e in p["entities"]),
     "messages": [{"role": "system", "content": SYS},
      {"role": "user", "content": 'EXISTING NODES:\n[{"id": "beacon", "label": "Beacon", "type": "Project"}]\n\nTOPIC HINT: same\nCURRENT TOPIC: beacon\n\nPREVIOUS ASSISTANT: Got it.\nUSER: We hired Sarah Chen as the new tech lead. Her salary is 180k/year.'}]},

    {"label": "valid relation (located_in)",
     "check": lambda p: any(r["relation"] in ["located_in", "part_of"] for r in p.get("relations", [])),
     "messages": [{"role": "system", "content": SYS},
      {"role": "user", "content": 'EXISTING NODES:\n[{"id": "cordoba", "label": "Córdoba", "type": "Place"}]\n\nTOPIC HINT: same\nCURRENT TOPIC: vivienda\n\nPREVIOUS ASSISTANT: Anotado.\nUSER: Vivimos en Güemes, Córdoba. Es un barrio tranquilo.'}]},

    {"label": "topic change",
     "check": lambda p: p["topic"]["action"] == "changed",
     "messages": [{"role": "system", "content": SYS},
      {"role": "user", "content": 'EXISTING NODES:\n[{"id": "beacon", "label": "Beacon", "type": "Project"}]\n\nTOPIC HINT: changed (medium confidence)\nCURRENT TOPIC: beacon\n\nPREVIOUS ASSISTANT: Understood.\nUSER: Let\'s switch topics. I want to talk about our hiring pipeline now.'}]},

    {"label": "numeric facts captured",
     "check": lambda p: len(p["facts"]) > 0 and any(c.isdigit() for f in p["facts"] for c in f["text"]),
     "messages": [{"role": "system", "content": SYS},
      {"role": "user", "content": 'EXISTING NODES:\n[{"id": "terreno_cipolletti", "label": "Terreno Cipolletti", "type": "Place", "facts": ["400m2 en zona residencial"]}]\n\nTOPIC HINT: same\nCURRENT TOPIC: compra terreno\n\nPREVIOUS ASSISTANT: Registrado.\nUSER: El terreno cuesta 32.000 USD y tiene orientación norte.'}]},

    {"label": "no invalid relations",
     "check": lambda p: all(r["relation"] in ["part_of", "created_by", "maintains", "works_at", "member_of", "uses_technology", "depends_on", "alternative_to", "located_in", "deployed_on", "produces", "serves", "documented_in", "participated_in", "triggered_by", "resulted_in"] for r in p.get("relations", [])),
     "messages": [{"role": "system", "content": SYS},
      {"role": "user", "content": 'EXISTING NODES:\n[{"id": "atlas", "label": "Atlas", "type": "Project", "facts": ["Microservices"]}]\n\nTOPIC HINT: same\nCURRENT TOPIC: atlas\n\nPREVIOUS ASSISTANT: OK.\nUSER: The project needs Redis for caching and was produced by the platform team.'}]},
]

passed = 0
for tc in test_cases:
    text = inner_tokenizer.apply_chat_template(
        tc["messages"], tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = inner_tokenizer(text, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
    response = inner_tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    print(f"--- {tc['label']} ---")
    try:
        parsed = json.loads(response)
        # V4 schema validation
        ok, schema_result = validate_s1_v4(response)
        if not ok:
            print(f"  [FAIL] Schema invalid: {schema_result}")
            continue
        check_ok = tc["check"](parsed)
        mark = "PASS" if check_ok else "FAIL"
        if check_ok:
            passed += 1
        print(f"  [{mark}] intent={parsed['intent']}, entities={len(parsed.get('entities',[]))}, "
              f"relations={len(parsed.get('relations',[]))}, facts={len(parsed.get('facts',[]))}")
        if not check_ok:
            print(f"  Output: {json.dumps(parsed, ensure_ascii=False)[:200]}")
    except json.JSONDecodeError as e:
        print(f"  [FAIL] JSON parse error: {e}")
        print(f"  Raw: {response[:200]}")

print(f"\nResult: {passed}/{len(test_cases)} passed")

Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model ready for inference.



c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`Attenti

RuntimeError: output with shape [1, 32, 1, 128] doesn't match the broadcast shape [1, 32, 977, 128]

## 6. Save Model

In [3]:
model.save_pretrained("./outputs/s1_sft_v4/final_lora")
tokenizer.save_pretrained("./outputs/s1_sft_v4/final_lora")
print("LoRA V4 saved to ./outputs/s1_sft_v4/final_lora")

LoRA V4 saved to ./outputs/s1_sft_v4/final_lora


In [ ]:
# Recover from checkpoint (run this if model was lost from VRAM)
from unsloth import FastLanguageModel
import torch
from pathlib import Path
import glob

MAX_SEQ_LENGTH = 2048

# Find latest checkpoint
checkpoints = sorted(glob.glob("./outputs/s1_sft_v4/checkpoint-*"))
if checkpoints:
    latest = checkpoints[-1]
    print(f"Loading from {latest}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=latest,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        dtype=None,
    )
    model.save_pretrained("./outputs/s1_sft_v4/final_lora")
    tokenizer.save_pretrained("./outputs/s1_sft_v4/final_lora")
    print("Saved to final_lora")
else:
    print("No checkpoints found!")

## 7. Merge LoRA + Export GGUF

In [4]:
import subprocess
from pathlib import Path

print("--- Merging LoRA into full model ---")
model.save_pretrained_merged(
    "./outputs/s1_sft_v4/merged",
    tokenizer,
    save_method="merged_16bit",
)
print("Merged model saved.\n")

src = Path("./outputs/s1_sft_v4/merged")
out = Path("./outputs/s1_sft_v4/gguf")
out.mkdir(exist_ok=True)

llama_dir = Path.home() / ".unsloth" / "llama.cpp"
quantize_bin = None
convert_script = None

for name in ["llama-quantize.exe", "quantize.exe"]:
    for d in [llama_dir, llama_dir / "build" / "bin" / "Release"]:
        p = d / name
        if p.exists():
            quantize_bin = p
            break
    if quantize_bin:
        break

for name in ["convert-hf-to-gguf.py", "convert_hf_to_gguf.py"]:
    p = llama_dir / name
    if p.exists():
        convert_script = p
        break

print(f"quantize: {quantize_bin}")
print(f"convert:  {convert_script}")
assert quantize_bin and quantize_bin.exists()
assert convert_script and convert_script.exists()

# BF16
print("\n--- Converting to BF16 GGUF ---")
bf16_path = out / "acervo-extractor-v4-bf16.gguf"
result = subprocess.run(
    ["python", str(convert_script), str(src),
     "--outfile", str(bf16_path), "--outtype", "bf16"],
    capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=600
)
if result.returncode != 0:
    print("ERROR:", result.stderr[-500:])
    raise RuntimeError("BF16 conversion failed")
print(f"BF16 GGUF: {bf16_path} ({bf16_path.stat().st_size / 1e9:.1f} GB)")

# Q4_K_M
print("\n--- Quantizing to Q4_K_M ---")
q4_path = out / "acervo-extractor-v4-Q4_K_M.gguf"
result = subprocess.run(
    [str(quantize_bin), str(bf16_path), str(q4_path), "Q4_K_M"],
    capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=600
)
if result.returncode != 0:
    print("ERROR:", result.stderr[-500:])
    raise RuntimeError("Quantization failed")

print(f"\nDone! Local GGUF ready:")
print(f"  BF16: {bf16_path} ({bf16_path.stat().st_size / 1e9:.1f} GB)")
print(f"  Q4_K_M: {q4_path} ({q4_path.stat().st_size / 1e9:.1f} GB)")
print(f"\nTest locally with: ollama create acervo-v4 -f Modelfile")

--- Merging LoRA into full model ---
Found HuggingFace hub cache directory: C:\Users\sandy\.cache\huggingface\hub


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00,  1.59it/s]


Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [04:06<00:00, 61.61s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [00:29<00:00,  7.44s/it]


Unsloth: Merge process complete. Saved to `d:\Development\acervo-graph-model\02_training\outputs\s1_sft_v4\merged`
Merged model saved.

quantize: C:\Users\sandy\.unsloth\llama.cpp\build\bin\Release\llama-quantize.exe
convert:  C:\Users\sandy\.unsloth\llama.cpp\convert_hf_to_gguf.py

--- Converting to BF16 GGUF ---
BF16 GGUF: outputs\s1_sft_v4\gguf\acervo-extractor-v4-bf16.gguf (16.4 GB)

--- Quantizing to Q4_K_M ---

Done! Local GGUF ready:
  BF16: outputs\s1_sft_v4\gguf\acervo-extractor-v4-bf16.gguf (16.4 GB)
  Q4_K_M: outputs\s1_sft_v4\gguf\acervo-extractor-v4-Q4_K_M.gguf (5.0 GB)

Test locally with: ollama create acervo-v4 -f Modelfile
